In [18]:
import torch
import torch.nn as nn
import numpy as np
import json
import io
import zipfile
from torch.utils.data import DataLoader, TensorDataset

In [19]:
# 1. Define the 1D-CNN Autoencoder Architecture
class SyscallAutoencoder1D(nn.Module):
    def __init__(self):
        super(SyscallAutoencoder1D, self).__init__()
        # Encoder (Compresses 64 -> 16)
        self.encoder = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2),  
            nn.Conv1d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2)   
        )
        # Decoder (Reconstructs 16 -> 64)
        self.decoder = nn.Sequential(
            nn.Upsample(scale_factor=2),
            nn.ConvTranspose1d(32, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Upsample(scale_factor=2),
            nn.ConvTranspose1d(16, 1, kernel_size=3, padding=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))

print("Architecture Loaded.")

Architecture Loaded.


In [21]:
def load_benign_heartbleed_data(main_zip_path: str, max_syscall_id: float = 332.0, window_size: int = 64):
    raw_benign_syscalls = []
    print(f"Extracting NORMAL baseline traces from: {main_zip_path}...")
    
    try:
        with zipfile.ZipFile(main_zip_path, 'r') as main_zip:
            
            training_zips = [
                f for f in main_zip.namelist() 
                if 'training/' in f and f.endswith('.zip') and '__MACOSX' not in f
            ]
            
            if not training_zips:
                print("ERROR: No nested training archives found.")
                return None, None
                
            for inner_zip_name in training_zips:
                inner_zip_bytes = io.BytesIO(main_zip.read(inner_zip_name))
                
                with zipfile.ZipFile(inner_zip_bytes, 'r') as inner_zip:
                    # 1. Target the .sc (SysCall) files instead of .json
                    for file_name in inner_zip.namelist():
                        if file_name.endswith('.sc') and '__MACOSX' not in file_name:
                            with inner_zip.open(file_name) as f:
                                # 2. Read the raw text and split it by whitespace/newlines
                                content = f.read().decode('utf-8', errors='ignore')
                                tokens = content.split()
                                
                                # 3. Convert text tokens directly into integers
                                for token in tokens:
                                    try:
                                        raw_benign_syscalls.append(int(token))
                                    except ValueError:
                                        pass # Skips any accidental string headers
                                        
    except FileNotFoundError:
        print(f"ERROR: Could not find {main_zip_path}.")
        return None, None

    if len(raw_benign_syscalls) < window_size:
        print("ERROR: Not enough data extracted.")
        return None, None

    print(f"\nExtraction Complete! Found {len(raw_benign_syscalls):,} normal system calls.")
    print("Slicing syscalls into sliding windows...")
    
    # --- Create Sliding Windows ---
    windows = []
    step_size = 16 
    
    for i in range(0, len(raw_benign_syscalls) - window_size + 1, step_size):
        windows.append(raw_benign_syscalls[i : i + window_size])
        
    windows_array = np.array(windows)
    
    # --- Normalize [0, 1] Range ---
    windows_normalized = windows_array / max_syscall_id
    windows_tensor = torch.tensor(windows_normalized, dtype=torch.float32).unsqueeze(1)
    
    return windows_tensor, len(raw_benign_syscalls)

# EXECUTE THE EXTRACTION PIPELINE
zip_filename = "CVE-2014-0160.zip"
windows_tensor, total_calls = load_benign_heartbleed_data(zip_filename)

if windows_tensor is not None:
    train_dataset = TensorDataset(windows_tensor)
    train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
    print(f"\nPipeline Ready.")
    print(f"Final Tensor Shape: {windows_tensor.shape}")

Extracting NORMAL baseline traces from: CVE-2014-0160.zip...

Extraction Complete! Found 2,943,700 normal system calls.
Slicing syscalls into sliding windows...

Pipeline Ready.
Final Tensor Shape: torch.Size([183978, 1, 64])
